# Kiểm tra môi trường Colab (GPU) cho train SASRec

Chạy bằng extension Colab trong VS Code: **Select Kernel → Colab → New Colab Server → chọn GPU (L4/A100/T4)**,
rồi **Run All**. Không cần upload dữ liệu — cell 3 đo tốc độ bằng dữ liệu ngẫu nhiên ĐÚNG kích thước job thật
(20K user, 6,16 triệu mẫu/epoch, 32.594 item, d=64, maxlen=50, batch 512, 50 mẫu âm).
Chạy xong nhớ **Save** để kết quả được ghi vào file.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
!free -h | head -2
!df -h /content | tail -1
!nproc

In [ ]:
import sys, torch, numpy, pandas
print("python", sys.version.split()[0], "| torch", torch.__version__, "| numpy", numpy.__version__, "| pandas", pandas.__version__)
print("CUDA:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHONG CO GPU — doi runtime sang GPU")

In [ ]:
# Do toc do train: cung kien truc + cach lay mau voi recsys_platform_sasrec.py
import time, torch, torch.nn as nn, torch.nn.functional as F
dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
N_ITEMS, D, L, B, NEG, N_SAMPLES = 32594, 64, 50, 512, 50, 6_158_408

class SASRec(nn.Module):
    def __init__(self):
        super().__init__()
        self.item_emb = nn.Embedding(N_ITEMS + 1, D, padding_idx=0)
        self.pos_emb = nn.Embedding(L, D)
        self.dropout = nn.Dropout(0.2)
        layer = nn.TransformerEncoderLayer(d_model=D, nhead=2, dim_feedforward=D * 4, dropout=0.2,
                                           batch_first=True, activation="gelu")
        self.encoder = nn.TransformerEncoder(layer, num_layers=1)
        self.ln = nn.LayerNorm(D)
    def forward(self, seqs):
        pos = torch.arange(seqs.size(1), device=seqs.device).unsqueeze(0)
        x = self.dropout(self.item_emb(seqs) + self.pos_emb(pos))
        mask = nn.Transformer.generate_square_subsequent_mask(seqs.size(1)).to(seqs.device)
        return self.ln(self.encoder(x, mask=mask, src_key_padding_mask=seqs == 0))

model = SASRec().to(dev)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
prob = torch.ones(N_ITEMS + 1, device=dev); prob[0] = 0

def step():
    x = torch.randint(1, N_ITEMS + 1, (B, L), device=dev)
    x[:, : L // 3] = 0  # co padding nhu du lieu that
    pos = torch.randint(1, N_ITEMS + 1, (B,), device=dev)
    h = model(x)[:, -1, :]
    neg = torch.multinomial(prob, B * NEG, replacement=True).view(B, NEG)
    logits = torch.einsum("bd,bnd->bn", h, model.item_emb(torch.cat([pos[:, None], neg], 1)))
    loss = F.cross_entropy(logits, torch.zeros(B, dtype=torch.long, device=dev))
    opt.zero_grad(); loss.backward(); opt.step()

for _ in range(20): step()            # khoi dong
if dev.type == "cuda": torch.cuda.synchronize()
t = time.perf_counter(); n = 200
for _ in range(n): step()
if dev.type == "cuda": torch.cuda.synchronize()
dt = (time.perf_counter() - t) / n
epoch_s = dt * N_SAMPLES / B
print(f"{dt*1000:.1f} ms/buoc | {B/dt:,.0f} mau/s | 1 epoch ~{epoch_s/60:.1f} phut | 30 epoch x 3 seed ~{epoch_s*90/3600:.1f} gio (tran tren, co dung som)")
if dev.type == "cuda": print(f"VRAM dinh: {torch.cuda.max_memory_allocated()/1e9:.2f} GB")